In [ ]:
import requests
import time
import random
import os
from google.colab import drive

# 1. Mount Google Drive to Colab
# When running this code, Colab will pop up a window requesting access to your Google Drive; please click allow.
drive.mount('/content/drive')

# 2. Set the save path to a dedicated folder in your Google Drive
# This ensures data is findable and won't be lost even if Colab disconnects
save_dir = '/content/drive/MyDrive/ECE_1513/Data/weather_data_toronto_city_center/raw'

# Check if the folder exists in Google Drive; if not, create it automatically
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"📁 Folder created in Google Drive: {save_dir}")
else:
    print(f"📁 Folder already exists, data will be saved to: {save_dir}")

# Environment Canada bulk download interface
base_url = "https://climate.weather.gc.ca/climate_data/bulk_data_e.html"

# Request headers
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive"
}

climate_id = "6158359"
years = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

print("\n🚀 Starting download and saving directly to Google Drive...")

for year in years:
    for month in range(1, 13):
        print(f"Requesting data for {year}-{month:02d}...")

        params = {
            "format": "csv",
            "climate_id": climate_id,
            "Year": year,
            "Month": month,
            "Day": 1,
            "time": "LST",
            "timeframe": 1,
            "submit": "Download Data"
        }

        filename = os.path.join(save_dir, f"weather_hourly_{climate_id}_{year}_{month:02d}.csv")

        try:
            response = requests.get(base_url, headers=headers, params=params, timeout=20)

            if response.status_code == 200:
                if response.text.strip().lower().startswith("<!doctype html>"):
                    print(f"❌ Warning: Request might be too fast and limited.")
                    time.sleep(10)
                    continue

                with open(filename, "wb") as f:
                    f.write(response.content)
                print(f"✅ {year}-{month:02d} downloaded successfully!")

            else:
                print(f"❌ Request failed, status code: {response.status_code}")

        except Exception as e:
            print(f"❌ Network anomaly: {e}")

        time.sleep(random.uniform(2.0, 4.0))

print(f"\n🎉 All downloads complete! Please check the 'Weather_Data_Archive' folder in the Colab file sidebar or via Google Drive web.")

In [ ]:
import pandas as pd
import glob
import os

# 1. Define your Google Drive folder path
folder_path = '/content/drive/MyDrive/ECE1513_Project/Data/weather_data_toronto_city_center/raw'

# Get the file paths for all hourly weather CSV files in the folder
# Ensure it matches the format we just downloaded
all_files = glob.glob(os.path.join(folder_path, "weather_hourly_*.csv"))

print(f"🔍 Found {len(all_files)} weather data files, starting merge...")

# 2. Physical merge: read all files and concatenate into one large DataFrame
df_list = []
for file in all_files:
    # Environment Canada data often has station info in the first few rows; the actual header is usually around row 16
    # However, the bulk_data_e interface usually handles this. Read directly; if it errors, add skiprows=16.
    temp_df = pd.read_csv(file, low_memory=False)
    df_list.append(temp_df)

# Concatenate 24 months of DataFrames
raw_merged_df = pd.concat(df_list, ignore_index=True)

# Save the uncleaned merged raw data as a complete CSV for physical backup
merged_csv_path = os.path.join(folder_path, "merged_weather_2020_2026_raw.csv")
raw_merged_df.to_csv(merged_csv_path, index=False)
print(f"💾 Raw merged file saved to: {merged_csv_path}")

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Ensure Google Drive is mounted (no side effects if already mounted in current session)
drive.mount('/content/drive')

# 2. Precisely configure input and output paths
input_path = '/content/drive/MyDrive/ECE1513_Project/Data/weather_data_toronto_city_center/merged_weather_2020_2026_raw.csv'
output_path = '/content/drive/MyDrive/ECE1513_Project/Data/weather_data_toronto_city_center/merged_weather_2020_2026_processed.csv'

print(f"📥 Reading raw weather data...\nPath: {input_path}")
# low_memory=False prevents warnings during type inference
raw_df = pd.read_csv(input_path, low_memory=False)

print("🧹 Starting feature cleaning and engineering...")

# 3. Filter core columns (matching real headers)
cols_to_keep = [
    'Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)',
    'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)',
    'Wind Dir (10s deg)', 'Visibility (km)', 'Rel Hum (%)', 'Weather'
]
clean_df = raw_df[[col for col in cols_to_keep if col in raw_df.columns]].copy()

# 4. Standardize timeline
# This step is crucial for subsequent time-series alignment (Join) with traffic data
clean_df['Date/Time (LST)'] = pd.to_datetime(clean_df['Date/Time (LST)'])
clean_df = clean_df.sort_values(by='Date/Time (LST)').reset_index(drop=True)

# 5. Handle missing values in continuous numerical columns (Forward Fill)
continuous_cols = ['Temp (°C)', 'Wind Spd (km/h)', 'Visibility (km)', 'Rel Hum (%)', 'Wind Dir (10s deg)']
for col in continuous_cols:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].ffill()

# 6. Handle missing precipitation values (Fill with 0)
if 'Precip. Amount (mm)' in clean_df.columns:
    clean_df['Precip. Amount (mm)'] = clean_df['Precip. Amount (mm)'].fillna(0)

# 7. [Feature Transformation] Cyclical encoding for wind direction (Sin / Cos)
if 'Wind Dir (10s deg)' in clean_df.columns:
    wind_rad = clean_df['Wind Dir (10s deg)'] * 10 * (np.pi / 180)
    clean_df['Wind_Sin'] = np.sin(wind_rad)
    clean_df['Wind_Cos'] = np.cos(wind_rad)
    clean_df = clean_df.drop(columns=['Wind Dir (10s deg)'])

# 8. [Feature Transformation] Extract weather states (One-Hot)
# These boolean features often have the most direct impact on road slipperiness and short-term congestion
if 'Weather' in clean_df.columns:
    clean_df['is_raining'] = clean_df['Weather'].str.contains('Rain|Drizzle', case=False, na=False).astype(int)
    clean_df['is_snowing'] = clean_df['Weather'].str.contains('Snow|Ice', case=False, na=False).astype(int)
    clean_df['is_foggy'] = clean_df['Weather'].str.contains('Fog|Mist', case=False, na=False).astype(int)
    clean_df = clean_df.drop(columns=['Weather'])

print("\n✨ Feature engineering complete! Preview of the model-ready DataFrame:")
display(clean_df.head())
print("\nBasic data info:")
display(clean_df.info())

# 9. Export as Processed CSV
print(f"\n💾 Saving processed data to...\n{output_path}")
# index=False ensures no redundant Unnamed: 0 index column is generated
clean_df.to_csv(output_path, index=False)
print("✅ Saved successfully!")

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set input and output paths
project_dir = '/content/drive/MyDrive/ECE1513_Project/Data/traffic_data'
if not os.path.exists(project_dir):
    os.makedirs(project_dir)

# This time, we only need the Speed table!
speed_data_path = os.path.join(project_dir, 'svc_raw_data_speed_2020_2024.csv')
output_traffic_path = os.path.join(project_dir, 'clean_traffic_2020_2024.csv')

print("📥 Reading traffic speed feature data...")
speed_cols = ['centreline_id', 'location_name', 'longitude', 'latitude', 'time_start', 'time_end', 'direction',
              'vol_1_19kph', 'vol_20_25kph', 'vol_26_30kph', 'vol_31_35kph', 'vol_36_40kph', 'vol_41_45kph',
              'vol_46_50kph', 'vol_51_55kph', 'vol_56_60kph', 'vol_61_65kph', 'vol_66_70kph', 'vol_71_75kph',
              'vol_76_80kph', 'vol_81_160kph']

# Read data
df = pd.read_csv(speed_data_path, usecols=speed_cols, low_memory=False)

print("⏳ Standardizing timestamps...")
df['time_start'] = pd.to_datetime(df['time_start'])
df['time_end'] = pd.to_datetime(df['time_end'])

print("🧮 Calculating [Total Traffic Volume] and [Average Speed] based on speed bins...")
speed_bins = [
    'vol_1_19kph', 'vol_20_25kph', 'vol_26_30kph', 'vol_31_35kph', 'vol_36_40kph',
    'vol_41_45kph', 'vol_46_50kph', 'vol_51_55kph', 'vol_56_60kph', 'vol_61_65kph',
    'vol_66_70kph', 'vol_71_75kph', 'vol_76_80kph', 'vol_81_160kph'
]
midpoints = np.array([10, 22.5, 28, 33, 38, 43, 48, 53, 58, 63, 68, 73, 78, 100])

# [Core Breakthrough 1]: Summing vehicle counts across all speed bins gives 15-min total volume!
df['volume_15min'] = df[speed_bins].sum(axis=1)

# [Core Breakthrough 2]: Calculate weighted average speed
weighted_sum_speed = (df[speed_bins] * midpoints).sum(axis=1)
df['avg_speed_kph'] = np.where(df['volume_15min'] > 0,
                               weighted_sum_speed / df['volume_15min'],
                               np.nan)

print("🧹 Cleaning data structure...")
final_cols = [
    'centreline_id', 'location_name', 'longitude', 'latitude', 'direction',
    'time_start', 'time_end', 'volume_15min', 'avg_speed_kph'
]
clean_traffic_df = df[final_cols].sort_values(by=['centreline_id', 'time_start']).reset_index(drop=True)

# Filter out records with zero vehicles in 15 mins (as congestion speed is irrelevant without cars)
# This helps remove noise for the model.
clean_traffic_df = clean_traffic_df[clean_traffic_df['volume_15min'] > 0].reset_index(drop=True)

print(f"\n💾 Saving to: {output_traffic_path}")
clean_traffic_df.to_csv(output_traffic_path, index=False)

print("\n✨ Traffic data processing complete! Data is available this time, previewing first 5 rows:")
display(clean_traffic_df.head())
print(f"\n✅ Successfully extracted {len(clean_traffic_df)} valid speed and volume records!")

In [ ]:
import pandas as pd
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configure file paths (Ensure paths match actual Drive locations)
base_data_dir = '/content/drive/MyDrive/ECE1513_Project/Data'

# Input file paths
weather_path = os.path.join(base_data_dir, 'weather_data_toronto_city_center', 'merged_weather_2020_2026_processed.csv')
traffic_path = os.path.join(base_data_dir, 'traffic_data', 'clean_traffic_2020_2024.csv')

# Final merged output path
output_path = os.path.join(base_data_dir, 'final_traffic_weather_merged.csv')

print("📥 Reading cleaned weather and traffic datasets...")
df_weather = pd.read_csv(weather_path, low_memory=False)
df_traffic = pd.read_csv(traffic_path, low_memory=False)

print(f"📊 Data read complete! Traffic table has {len(df_traffic)} rows, Weather table has {len(df_weather)} rows.")

# 3. Standardize time formats
print("⏳ Unifying time series formats...")
# Weather timestamps
df_weather['Date/Time (LST)'] = pd.to_datetime(df_weather['Date/Time (LST)'])
# Traffic timestamps
df_traffic['time_start'] = pd.to_datetime(df_traffic['time_start'])
df_traffic['time_end'] = pd.to_datetime(df_traffic['time_end'])

# 4. [Core Alignment Logic] Time Flooring
# Floor traffic time_start (e.g., 15:15:00) to the nearest hour (15:00:00) for matching
print("🔗 Aligning and merging data by [Hour]...")
df_traffic['join_time_hour'] = df_traffic['time_start'].dt.floor('h')

# Use Left Join: Keep all traffic records and append matching weather features
merged_df = pd.merge(
    df_traffic,
    df_weather,
    left_on='join_time_hour',
    right_on='Date/Time (LST)',
    how='left'
)

# 5. Clean redundant columns and handle potential missing values
print("🧹 Cleaning merged table structure...")
# Drop temporary join column and duplicated date columns from weather table
columns_to_drop = ['join_time_hour', 'Date/Time (LST)', 'Year', 'Month', 'Day', 'Time (LST)']
merged_df = merged_df.drop(columns=[col for col in columns_to_drop if col in merged_df.columns])

# Since traffic data goes to 2024 and weather to 2026 (or potential missing hours),
# use forward fill on weather features to ensure no NaN values disrupt training
weather_features = [
    'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)', 'Visibility (km)',
    'Rel Hum (%)', 'Wind_Sin', 'Wind_Cos', 'is_raining', 'is_snowing', 'is_foggy'
]
for col in weather_features:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].ffill()

# 6. Save final merged table
print(f"\n💾 Saving final merged dataset to:\n{output_path}")
merged_df.to_csv(output_path, index=False)

print("\n✨ Perfect merge! Preview of final baseline table:")
display(merged_df.head())
print(f"\n✅ Successfully generated merged dataset with {len(merged_df)} rows.")

In [ ]:
import pandas as pd

# 1. Configure file path (replace with actual path of merged table in Google Drive)
file_path = '/content/drive/MyDrive/ECE1513_Project/Data/final_traffic_weather_merged.csv'

print("📥 Reading dataset...")
df = pd.read_csv(file_path, low_memory=False)

# Ensure time_start is datetime format
df['time_start'] = pd.to_datetime(df['time_start'])

# ---------------------------------------------------------
# Goal 1: Count unique locations
# ---------------------------------------------------------
unique_locations = df['centreline_id'].nunique()
print(f"\n✅ Total of {unique_locations} unique road segments (Locations) in the dataset.")

# ---------------------------------------------------------
# Goal 2: Calculate effective data duration per location
# ---------------------------------------------------------
print("🧮 Calculating effective data duration per segment (in hours)...")

# Floor 15-min timestamps to hour level (e.g., 00:15 and 00:30 both become 00:00)
df['hour_trunc'] = df['time_start'].dt.floor('h')

# Group by centreline_id and location_name, count [unique hour_trunc]
hours_per_location = df.groupby(['centreline_id', 'location_name'])['hour_trunc'].nunique().reset_index()

# Rename columns for clarity
hours_per_location.rename(columns={'hour_trunc': 'total_hours_of_data'}, inplace=True)

# Add helper column: convert hours to "days" (assuming 24h/day recording) for easier reading
hours_per_location['approx_days'] = (hours_per_location['total_hours_of_data'] / 24).round(1)

# Sort by duration descending
hours_per_location = hours_per_location.sort_values(by='total_hours_of_data', ascending=False).reset_index(drop=True)

print("\n📊 Data duration stats per Location (Top 20):")
display(hours_per_location.head(20))

print("\n📉 Data duration stats per Location (Bottom 10):")
display(hours_per_location.tail(10))

# ---------------------------------------------------------
# Goal 3: Global data distribution overview
# ---------------------------------------------------------
print("\n🔍 Data distribution overview:")
print(f"Longest segment has: {hours_per_location['total_hours_of_data'].max()} hours (approx {hours_per_location['approx_days'].max()} days) of data")
print(f"Shortest segment has: {hours_per_location['total_hours_of_data'].min()} hours (approx {hours_per_location['approx_days'].min()} days) of data")
print(f"Median data volume: {hours_per_location['total_hours_of_data'].median()} hours (approx {round(hours_per_location['total_hours_of_data'].median()/24, 1)} days)")

# Save stats result for reports or EDA
summary_path = '/content/drive/MyDrive/ECE1513_Project/Data/location_data_summary.csv'
hours_per_location.to_csv(summary_path, index=False)
print(f"\n💾 Detailed stats saved to: {summary_path}")

In [ ]:
#checking data quality
#may remove location less than 3 days

import pandas as pd

# Read merged table
file_path = '/content/drive/MyDrive/ECE1513_Project/Data/final_traffic_weather_merged.csv'
print("📥 Reading dataset to calculate loss rate...")
df = pd.read_csv(file_path, low_memory=False)
df['time_start'] = pd.to_datetime(df['time_start'])

# Basic stats
total_rows = len(df)
total_locations = df['centreline_id'].nunique()

# Calculate effective hours per segment
df['hour_trunc'] = df['time_start'].dt.floor('h')
hours_per_loc = df.groupby('centreline_id')['hour_trunc'].nunique()

def calculate_loss(min_hours, days_label):
    # Identify qualified segments
    valid_locs = hours_per_loc[hours_per_loc >= min_hours].index

    # Filter data
    retained_df = df[df['centreline_id'].isin(valid_locs)]
    retained_rows = len(retained_df)
    retained_locations = len(valid_locs)

    # Calculate loss rates
    lost_rows = total_rows - retained_rows
    lost_locs = total_locations - retained_locations

    row_loss_pct = (lost_rows / total_rows) * 100
    loc_loss_pct = (lost_locs / total_locations) * 100

    print(f"\n=======================================")
    print(f" 🛑 Plan: Filter out data with less than {days_label} ({min_hours} hours)")
    print(f"=======================================")
    print(f"📍 Lost segments: {lost_locs} ({loc_loss_pct:.1f}%) -> Remaining {retained_locations} segments")
    print(f"📊 Lost data rows: {lost_rows:,} ({row_loss_pct:.1f}%) -> Remaining {retained_rows:,} rows")

# Calculate losses for 3 days (72h) and 7 days (168h)
calculate_loss(72, "3 Days")
calculate_loss(168, "7 Days")

In [ ]:
# ============================================================
# ECE1513 Project — XGBoost Regression Model
# Improvements over RF baseline:
#   - Speed deviation feature (fixes leakage)
#   - Deviation-based lags & rolling stats
#   - XGBoost with early stopping
#   - Saves model with joblib
# ============================================================

import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import drive

# ── 1. Load Data ─────────────────────────────────────────────

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/ECE1513_Project/Data/final_traffic_weather_merged.csv'

df = pd.read_csv(file_path)
df['time_start'] = pd.to_datetime(df['time_start'])
df['time_end']   = pd.to_datetime(df['time_end'])
df = df.sort_values(['centreline_id', 'direction', 'time_start']).reset_index(drop=True)

print(f"Total records loaded: {len(df)}")
print(f"Date range: {df['time_start'].min()} to {df['time_start'].max()}")


# ── 2. Time Features ─────────────────────────────────────────
df['hour']       = df['time_start'].dt.hour
df['minute']     = df['time_start'].dt.minute
df['dayofweek']  = df['time_start'].dt.dayofweek
df['month']      = df['time_start'].dt.month
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['dayofweek'] / 7)


# ── 3. Speed Deviation (leakage fix) ─────────────────────────
# For each road segment + direction + hour + day-of-week,
# compute the historical average speed. Then subtract it.
# This tells the model "is traffic abnormally slow RIGHT NOW?"
# instead of leaking the raw speed value directly into lags.

historical_avg = (
    df.groupby(['centreline_id', 'direction', 'hour', 'dayofweek'])['avg_speed_kph']
    .transform('mean')
)
df['speed_deviation'] = df['avg_speed_kph'] - historical_avg

# Same for volume
historical_vol_avg = (
    df.groupby(['centreline_id', 'direction', 'hour', 'dayofweek'])['volume_15min']
    .transform('mean')
)
df['volume_deviation'] = df['volume_15min'] - historical_vol_avg


# ── 4. Lag Features (on deviation, not raw speed) ────────────
group_keys = ['centreline_id', 'direction']

# Deviation lags — these are the safe, non-leaky version
for lag in [1, 2, 4, 8]:
    df[f'speed_dev_lag_{lag}']  = df.groupby(group_keys)['speed_deviation'].shift(lag)
    df[f'volume_dev_lag_{lag}'] = df.groupby(group_keys)['volume_deviation'].shift(lag)

# Keep raw speed lags too but with a larger minimum gap (lag >= 2)
# lag_1 of raw speed is fine as long as we don't also include roll_mean_1
for lag in [2, 4, 8]:
    df[f'speed_lag_{lag}']  = df.groupby(group_keys)['avg_speed_kph'].shift(lag)
    df[f'volume_lag_{lag}'] = df.groupby(group_keys)['volume_15min'].shift(lag)


# ── 5. Rolling Window Features (on deviation) ─────────────────
for window in [4, 8]:
    grp = df.groupby(group_keys)

    # Rolling mean of deviation — "has it been getting worse recently?"
    df[f'speed_dev_roll_mean_{window}'] = (
        grp['speed_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f'speed_dev_roll_std_{window}'] = (
        grp['speed_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )
    df[f'volume_dev_roll_mean_{window}'] = (
        grp['volume_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

    # Raw speed rolling (shifted by 2 to avoid leakage from t-1)
    df[f'speed_roll_mean_{window}'] = (
        grp['avg_speed_kph']
        .transform(lambda x: x.shift(2).rolling(window, min_periods=1).mean())
    )


# ── 6. Historical context features ───────────────────────────
# Give the model direct access to what "normal" looks like
# for this segment/hour/DOW combination
df['hist_avg_speed']  = historical_avg
df['hist_avg_volume'] = historical_vol_avg


# ── 7. Define Features & Target ──────────────────────────────
TARGET = 'avg_speed_kph'

FEATURES = [
    # Location
    'longitude', 'latitude', 'centreline_id',
    # Time
    'hour', 'dayofweek', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    # Weather
    'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)',
    'Visibility (km)', 'Rel Hum (%)',
    'Wind_Sin', 'Wind_Cos',
    'is_raining', 'is_snowing', 'is_foggy',
    # Historical context (what is "normal" here)
    'hist_avg_speed', 'hist_avg_volume',
    # Deviation lags (core non-leaky signal)
    'speed_dev_lag_1', 'speed_dev_lag_2', 'speed_dev_lag_4', 'speed_dev_lag_8',
    'volume_dev_lag_1', 'volume_dev_lag_2', 'volume_dev_lag_4', 'volume_dev_lag_8',
    # Deviation rolling stats
    'speed_dev_roll_mean_4', 'speed_dev_roll_mean_8',
    'speed_dev_roll_std_4',  'speed_dev_roll_std_8',
    'volume_dev_roll_mean_4', 'volume_dev_roll_mean_8',
    # Raw lags (lag >= 2 only)
    'speed_lag_2', 'speed_lag_4', 'speed_lag_8',
    'volume_lag_2', 'volume_lag_4', 'volume_lag_8',
    # Raw rolling (shifted by 2)
    'speed_roll_mean_4', 'speed_roll_mean_8',
    # Raw volume (current interval)
    'volume_15min',
]

df_model = df.dropna(subset=FEATURES + [TARGET]).copy()
print(f"\nRecords after dropping NaN: {len(df_model)}")


# ── 8. Time-Aware Train / Val / Test Split ───────────────────
# 70% train | 10% validation (for early stopping) | 20% test
# Strict time ordering — no shuffling

n = len(df_model)
sorted_df = df_model.sort_values('time_start').reset_index(drop=True)

train_end = int(n * 0.70)
val_end   = int(n * 0.80)

train_df = sorted_df.iloc[:train_end]
val_df   = sorted_df.iloc[train_end:val_end]
test_df  = sorted_df.iloc[val_end:]

print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")
print(f"Train time range: {train_df['time_start'].min()} to {train_df['time_start'].max()}")
print(f"Test time range:  {test_df['time_start'].min()} to {test_df['time_start'].max()}")

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_val   = val_df[FEATURES]
y_val   = val_df[TARGET]
X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]


# ── 9. Train XGBoost ─────────────────────────────────────────
print("\nTraining XGBoost...")

xgb_model = XGBRegressor(
    n_estimators      = 1000,   # high ceiling — early stopping will cut this down
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_weight  = 5,
    subsample         = 0.8,    # row sampling per tree (reduces overfitting)
    colsample_bytree  = 0.8,    # feature sampling per tree
    reg_alpha         = 0.1,    # L1 regularization
    reg_lambda        = 1.0,    # L2 regularization
    random_state      = 42,
    n_jobs            = -1,
    tree_method       = 'hist', # fast histogram method — good for large datasets
    early_stopping_rounds = 30, # stop if val loss doesn't improve for 30 rounds
    eval_metric       = 'rmse',
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,   # print every 50 rounds
)

print(f"\nBest iteration: {xgb_model.best_iteration}")


# ── 10. Evaluate ─────────────────────────────────────────────
y_pred = xgb_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n===== XGBoost Model Performance =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R^2  : {r2:.4f}")


# ── 11. Save Model & Predictions ─────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/ECE1513_Project/Models/xgb_model.pkl'
joblib.dump(xgb_model, MODEL_PATH)
print(f"\nSaved model to {MODEL_PATH}")

preds_df = test_df[['centreline_id', 'direction', 'time_start', TARGET]].copy()
preds_df['predicted_speed'] = y_pred
preds_df.to_csv(
    '/content/drive/MyDrive/ECE1513_Project/Data/xgb_predictions.csv',
    index=False
)
print("Saved predictions to xgb_predictions.csv")


# ── 12. Feature Importance ───────────────────────────────────
importance_df = pd.DataFrame({
    'feature':    FEATURES,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False)

print("\nTop 20 important features:")
print(importance_df.head(20).to_string())

importance_df.to_csv(
    '/content/drive/MyDrive/ECE1513_Project/Data/xgb_feature_importance.csv',
    index=False
)
print("\nSaved feature importance to xgb_feature_importance.csv")


# ── 13. Load & Reuse (example) ───────────────────────────────
# To reuse this model in any future notebook, just run:
#
#   import joblib
#   xgb_model = joblib.load('/content/drive/MyDrive/ECE1513_Project/Models/xgb_model.pkl')
#   y_pred = xgb_model.predict(X_test)

In [ ]:
# ============================================================
# ECE1513 Project — XGBoost Regression Model
# Improvements over RF baseline:
#   - Speed deviation feature (fixes leakage)
#   - Deviation-based lags & rolling stats
#   - XGBoost with early stopping
#   - Saves model with joblib
# ============================================================

import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import drive

# ── 1. Load Data ─────────────────────────────────────────────

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/ECE1513_Project/Data/final_traffic_weather_merged.csv'

df = pd.read_csv(file_path)
df['time_start'] = pd.to_datetime(df['time_start'])
df['time_end']   = pd.to_datetime(df['time_end'])
df = df.sort_values(['centreline_id', 'direction', 'time_start']).reset_index(drop=True)

print(f"Total records loaded: {len(df)}")
print(f"Date range: {df['time_start'].min()} to {df['time_start'].max()}")


# ── 2. Time Features ─────────────────────────────────────────
df['hour']       = df['time_start'].dt.hour
df['minute']     = df['time_start'].dt.minute
df['dayofweek']  = df['time_start'].dt.dayofweek
df['month']      = df['time_start'].dt.month
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['dayofweek'] / 7)


# ── 3. Speed Deviation (leakage fix) ─────────────────────────
# For each road segment + direction + hour + day-of-week,
# compute the historical average speed. Then subtract it.
# This tells the model "is traffic abnormally slow RIGHT NOW?"
# instead of leaking the raw speed value directly into lags.

historical_avg = (
    df.groupby(['centreline_id', 'direction', 'hour', 'dayofweek'])['avg_speed_kph']
    .transform('mean')
)
df['speed_deviation'] = df['avg_speed_kph'] - historical_avg

# Same for volume
historical_vol_avg = (
    df.groupby(['centreline_id', 'direction', 'hour', 'dayofweek'])['volume_15min']
    .transform('mean')
)
df['volume_deviation'] = df['volume_15min'] - historical_vol_avg


# ── 4. Lag Features (on deviation, not raw speed) ────────────
group_keys = ['centreline_id', 'direction']

# Deviation lags — these are the safe, non-leaky version
for lag in [1, 2, 4, 8]:
    df[f'speed_dev_lag_{lag}']  = df.groupby(group_keys)['speed_deviation'].shift(lag)
    df[f'volume_dev_lag_{lag}'] = df.groupby(group_keys)['volume_deviation'].shift(lag)

# Keep raw speed lags too but with a larger minimum gap (lag >= 2)
# lag_1 of raw speed is fine as long as we don't also include roll_mean_1
for lag in [2, 4, 8]:
    df[f'speed_lag_{lag}']  = df.groupby(group_keys)['avg_speed_kph'].shift(lag)
    df[f'volume_lag_{lag}'] = df.groupby(group_keys)['volume_15min'].shift(lag)


# ── 5. Rolling Window Features (on deviation) ─────────────────
for window in [4, 8]:
    grp = df.groupby(group_keys)

    # Rolling mean of deviation — "has it been getting worse recently?"
    df[f'speed_dev_roll_mean_{window}'] = (
        grp['speed_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f'speed_dev_roll_std_{window}'] = (
        grp['speed_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )
    df[f'volume_dev_roll_mean_{window}'] = (
        grp['volume_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

    # Raw speed rolling (shifted by 2 to avoid leakage from t-1)
    df[f'speed_roll_mean_{window}'] = (
        grp['avg_speed_kph']
        .transform(lambda x: x.shift(2).rolling(window, min_periods=1).mean())
    )


# ── 6. Historical context features ───────────────────────────
# Give the model direct access to what "normal" looks like
# for this segment/hour/DOW combination
df['hist_avg_speed']  = historical_avg
df['hist_avg_volume'] = historical_vol_avg


# ── 7. Define Features & Target ──────────────────────────────
TARGET = 'speed_deviation'

FEATURES = [
    # Location
    'longitude', 'latitude', 'centreline_id',
    # Time
    'hour', 'dayofweek', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    # Weather
    'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)',
    'Visibility (km)', 'Rel Hum (%)',
    'Wind_Sin', 'Wind_Cos',
    'is_raining', 'is_snowing', 'is_foggy',
    # Historical context (what is "normal" here)
    'hist_avg_speed', 'hist_avg_volume',
    # Deviation lags (core non-leaky signal)
    'speed_dev_lag_1', 'speed_dev_lag_2', 'speed_dev_lag_4', 'speed_dev_lag_8',
    'volume_dev_lag_1', 'volume_dev_lag_2', 'volume_dev_lag_4', 'volume_dev_lag_8',
    # Deviation rolling stats
    'speed_dev_roll_mean_4', 'speed_dev_roll_mean_8',
    'speed_dev_roll_std_4',  'speed_dev_roll_std_8',
    'volume_dev_roll_mean_4', 'volume_dev_roll_mean_8',
    # Raw lags (lag >= 2 only)
    'speed_lag_2', 'speed_lag_4', 'speed_lag_8',
    'volume_lag_2', 'volume_lag_4', 'volume_lag_8',
    # Raw volume (current interval)
    'volume_15min',
]

df_model = df.dropna(subset=FEATURES + [TARGET]).copy()
print(f"\nRecords after dropping NaN: {len(df_model)}")


# ── 8. Time-Aware Train / Val / Test Split ───────────────────
# 70% train | 10% validation (for early stopping) | 20% test
# Strict time ordering — no shuffling

n = len(df_model)
sorted_df = df_model.sort_values('time_start').reset_index(drop=True)

train_end = int(n * 0.70)
val_end   = int(n * 0.80)

train_df = sorted_df.iloc[:train_end]
val_df   = sorted_df.iloc[train_end:val_end]
test_df  = sorted_df.iloc[val_end:]

print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")
print(f"Train time range: {train_df['time_start'].min()} to {train_df['time_start'].max()}")
print(f"Test time range:  {test_df['time_start'].min()} to {test_df['time_start'].max()}")

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_val   = val_df[FEATURES]
y_val   = val_df[TARGET]
X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]


# ── 9. Train XGBoost ─────────────────────────────────────────
print("\nTraining XGBoost...")

xgb_model = XGBRegressor(
    n_estimators      = 1000,   # high ceiling — early stopping will cut this down
    learning_rate     = 0.05,
    max_depth         = 6,
    min_child_weight  = 5,
    subsample         = 0.8,    # row sampling per tree (reduces overfitting)
    colsample_bytree  = 0.8,    # feature sampling per tree
    reg_alpha         = 0.1,    # L1 regularization
    reg_lambda        = 1.0,    # L2 regularization
    random_state      = 42,
    n_jobs            = -1,
    tree_method       = 'hist', # fast histogram method — good for large datasets
    early_stopping_rounds = 30, # stop if val loss doesn't improve for 30 rounds
    eval_metric       = 'rmse',
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,   # print every 50 rounds
)

print(f"\nBest iteration: {xgb_model.best_iteration}")


# ── 10. Evaluate ─────────────────────────────────────────────
y_pred = xgb_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n===== XGBoost Model Performance =====")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R^2  : {r2:.4f}")


# ── 11. Save Model & Predictions ─────────────────────────────
MODEL_PATH = '/content/drive/MyDrive/ECE1513_Project/Models/xgb_model.pkl'
joblib.dump(xgb_model, MODEL_PATH)
print(f"\nSaved model to {MODEL_PATH}")

preds_df = test_df[['centreline_id', 'direction', 'time_start', TARGET]].copy()
preds_df['predicted_speed'] = y_pred
preds_df.to_csv(
    '/content/drive/MyDrive/ECE1513_Project/Data/xgb_predictions.csv',
    index=False
)
print("Saved predictions to xgb_predictions.csv")


# ── 12. Feature Importance ───────────────────────────────────
importance_df = pd.DataFrame({
    'feature':    FEATURES,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False)

print("\nTop 20 important features:")
print(importance_df.head(20).to_string())

importance_df.to_csv(
    '/content/drive/MyDrive/ECE1513_Project/Data/xgb_feature_importance.csv',
    index=False
)
print("\nSaved feature importance to xgb_feature_importance.csv")


# ── 13. Load & Reuse (example) ───────────────────────────────
# To reuse this model in any future notebook, just run:
#
#   import joblib
#   xgb_model = joblib.load('/content/drive/MyDrive/ECE1513_Project/Models/xgb_model.pkl')
#   y_pred = xgb_model.predict(X_test)

In [ ]:
# ============================================================
# ECE1513 Project — XGBoost Regression Model
# Target: avg_speed_kph (raw speed, not deviation)
# Leakage fixes:
#   1. Historical averages computed ONLY from training data
#   2. Split BEFORE computing deviation features
#   3. Lags & rolling windows use shift() to avoid peeking
#   4. No current-interval raw speed used as a feature
# ============================================================

import pandas as pd
import numpy as np
import joblib
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import drive

# ── 1. Load Data ─────────────────────────────────────────────

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/ECE1513_Project/Data/final_traffic_weather_ml_ready.csv'

df = pd.read_csv(file_path)
df['time_start'] = pd.to_datetime(df['time_start'])
df['time_end']   = pd.to_datetime(df['time_end'])
df = df.sort_values(['centreline_id', 'direction', 'time_start']).reset_index(drop=True)

print(f"Total records loaded: {len(df)}")
print(f"Date range: {df['time_start'].min()} to {df['time_start'].max()}")


# ── 2. Time Features ─────────────────────────────────────────

df['hour']       = df['time_start'].dt.hour
df['minute']     = df['time_start'].dt.minute
df['dayofweek']  = df['time_start'].dt.dayofweek
df['month']      = df['time_start'].dt.month
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin']  = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos']  = np.cos(2 * np.pi * df['dayofweek'] / 7)


# ── 3. Lag & Rolling Features (built before split — safe) ────
# These only look BACKWARD via shift(), so no future leakage.
# We build them on the full sorted dataframe so that the first
# rows of val/test can still reference recent train-period lags.

group_keys = ['centreline_id', 'direction']

# Raw speed lags (shift >= 1 is safe when predicting speed)
for lag in [1, 2, 4, 8]:
    df[f'speed_lag_{lag}']  = df.groupby(group_keys)['avg_speed_kph'].shift(lag)
    df[f'volume_lag_{lag}'] = df.groupby(group_keys)['volume_15min'].shift(lag)

# Rolling windows on raw speed & volume (shifted by 1 first)
for window in [4, 8]:
    grp = df.groupby(group_keys)

    df[f'speed_roll_mean_{window}'] = (
        grp['avg_speed_kph']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f'speed_roll_std_{window}'] = (
        grp['avg_speed_kph']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )
    df[f'volume_roll_mean_{window}'] = (
        grp['volume_15min']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )


# ── 4. Time-Aware Train / Val / Test Split ───────────────────
# Split BEFORE computing historical averages to prevent leakage.
# 70% train | 10% validation (early stopping) | 20% test

sorted_df = df.sort_values('time_start').reset_index(drop=True)
n = len(sorted_df)

train_end = int(n * 0.70)
val_end   = int(n * 0.80)

train_df = sorted_df.iloc[:train_end].copy()
val_df   = sorted_df.iloc[train_end:val_end].copy()
test_df  = sorted_df.iloc[val_end:].copy()

print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")
print(f"Train period: {train_df['time_start'].min()} → {train_df['time_start'].max()}")
print(f"Val period:   {val_df['time_start'].min()} → {val_df['time_start'].max()}")
print(f"Test period:  {test_df['time_start'].min()} → {test_df['time_start'].max()}")


# ── 5. Historical Averages (train-only, no leakage) ──────────
# Compute the "normal" speed/volume for each segment+direction+
# hour+dayofweek using ONLY training data. Then map onto all splits.

hist_keys = ['centreline_id', 'direction', 'hour', 'dayofweek']

train_hist = (
    train_df
    .groupby(hist_keys)
    .agg(
        hist_avg_speed=('avg_speed_kph', 'mean'),
        hist_avg_volume=('volume_15min', 'mean'),
    )
    .reset_index()
)

# Merge train-only averages onto every split
for split_df in [train_df, val_df, test_df]:
    # Drop columns if they already exist (safety for re-runs)
    split_df.drop(columns=['hist_avg_speed', 'hist_avg_volume'],
                  errors='ignore', inplace=True)

    merged = split_df.merge(train_hist, on=hist_keys, how='left')
    split_df['hist_avg_speed']  = merged['hist_avg_speed'].values
    split_df['hist_avg_volume'] = merged['hist_avg_volume'].values

# For val/test rows with unseen segment+hour+dow combos, fill with
# a broader average (segment+direction level) so we don't get NaNs.
train_segment_avg = (
    train_df
    .groupby(['centreline_id', 'direction'])
    .agg(
        fallback_speed=('avg_speed_kph', 'mean'),
        fallback_volume=('volume_15min', 'mean'),
    )
    .reset_index()
)

fallback_speed_map  = train_segment_avg.set_index(['centreline_id', 'direction'])['fallback_speed']
fallback_volume_map = train_segment_avg.set_index(['centreline_id', 'direction'])['fallback_volume']

for split_df in [val_df, test_df]:
    mask_speed = split_df['hist_avg_speed'].isna()
    mask_vol   = split_df['hist_avg_volume'].isna()

    if mask_speed.any():
        fb_keys = list(zip(split_df.loc[mask_speed, 'centreline_id'],
                           split_df.loc[mask_speed, 'direction']))
        split_df.loc[mask_speed, 'hist_avg_speed'] = [
            fallback_speed_map.get(k, np.nan) for k in fb_keys
        ]

    if mask_vol.any():
        fb_keys = list(zip(split_df.loc[mask_vol, 'centreline_id'],
                           split_df.loc[mask_vol, 'direction']))
        split_df.loc[mask_vol, 'hist_avg_volume'] = [
            fallback_volume_map.get(k, np.nan) for k in fb_keys
        ]

print(f"\nHistorical avg NaNs in test: "
      f"speed={test_df['hist_avg_speed'].isna().sum()}, "
      f"volume={test_df['hist_avg_volume'].isna().sum()}")


# ── 6. Deviation Features (using train-only averages) ────────
# These are INPUT FEATURES, not the target. They tell the model
# how recent traffic compares to what's "normal" for this slot.

for split_df in [train_df, val_df, test_df]:
    split_df['speed_deviation']  = split_df['avg_speed_kph'] - split_df['hist_avg_speed']
    split_df['volume_deviation'] = split_df['volume_15min']  - split_df['hist_avg_volume']

# Deviation lags (already have raw lags; now add deviation lags)
# These were built from the full df above. We need to recompute
# them per split since deviation itself depends on train-only avgs.
# However, since lags cross the split boundary, the cleanest approach
# is to concatenate, compute, then re-split.

combined = pd.concat([train_df, val_df, test_df], ignore_index=True)
combined = combined.sort_values(['centreline_id', 'direction', 'time_start']).reset_index(drop=True)

for lag in [1, 2, 4, 8]:
    combined[f'speed_dev_lag_{lag}']  = combined.groupby(group_keys)['speed_deviation'].shift(lag)
    combined[f'volume_dev_lag_{lag}'] = combined.groupby(group_keys)['volume_deviation'].shift(lag)

for window in [4, 8]:
    grp = combined.groupby(group_keys)
    combined[f'speed_dev_roll_mean_{window}'] = (
        grp['speed_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    combined[f'speed_dev_roll_std_{window}'] = (
        grp['speed_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )
    combined[f'volume_dev_roll_mean_{window}'] = (
        grp['volume_deviation']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

# Re-split using time boundaries (same cutoffs as before)
train_df = combined[combined['time_start'] <= train_df['time_start'].max()].copy()
val_df   = combined[
    (combined['time_start'] > train_df['time_start'].max()) &
    (combined['time_start'] <= test_df['time_start'].min())
].copy()

# Use original test start as boundary
test_start = sorted_df.iloc[val_end]['time_start']
val_df  = combined[
    (combined['time_start'] > sorted_df.iloc[train_end - 1]['time_start']) &
    (combined['time_start'] <= sorted_df.iloc[val_end - 1]['time_start'])
].copy()
test_df = combined[
    combined['time_start'] > sorted_df.iloc[val_end - 1]['time_start']
].copy()

print(f"\nAfter deviation features:")
print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


# ── 7. Define Features & Target ──────────────────────────────

TARGET = 'avg_speed_kph'

FEATURES = [
    # Location
    'longitude', 'latitude', 'centreline_id',

    # Time (raw + cyclical)
    'hour', 'dayofweek', 'month', 'is_weekend',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',

    # Weather
    'Temp (°C)', 'Precip. Amount (mm)', 'Wind Spd (km/h)',
    'Visibility (km)', 'Rel Hum (%)',
    'Wind_Sin', 'Wind_Cos',
    'is_raining', 'is_snowing', 'is_foggy',

    # Historical context (computed from train only)
    'hist_avg_speed', 'hist_avg_volume',

    # Raw speed lags
    'speed_lag_1', 'speed_lag_2', 'speed_lag_4', 'speed_lag_8',

    # Raw volume lags
    'volume_lag_1', 'volume_lag_2', 'volume_lag_4', 'volume_lag_8',

    # Raw rolling stats
    'speed_roll_mean_4', 'speed_roll_mean_8',
    'speed_roll_std_4',  'speed_roll_std_8',
    'volume_roll_mean_4', 'volume_roll_mean_8',

    # Deviation lags (based on train-only historical avg)
    'speed_dev_lag_1', 'speed_dev_lag_2', 'speed_dev_lag_4', 'speed_dev_lag_8',
    'volume_dev_lag_1', 'volume_dev_lag_2', 'volume_dev_lag_4', 'volume_dev_lag_8',

    # Deviation rolling stats
    'speed_dev_roll_mean_4', 'speed_dev_roll_mean_8',
    'speed_dev_roll_std_4',  'speed_dev_roll_std_8',
    'volume_dev_roll_mean_4', 'volume_dev_roll_mean_8',

    # Current-interval volume (available in real-time from sensors)
    'volume_15min',
]

# Drop rows with NaN in features or target
for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    before = len(split_df)
    mask = split_df[FEATURES + [TARGET]].notna().all(axis=1)
    if name == 'train':
        train_df = split_df[mask].copy()
    elif name == 'val':
        val_df = split_df[mask].copy()
    else:
        test_df = split_df[mask].copy()
    print(f"  {name}: {before} → {mask.sum()} (dropped {before - mask.sum()} NaN rows)")

X_train = train_df[FEATURES]
y_train = train_df[TARGET]
X_val   = val_df[FEATURES]
y_val   = val_df[TARGET]
X_test  = test_df[FEATURES]
y_test  = test_df[TARGET]


# ── 8. Train XGBoost ─────────────────────────────────────────

print("\nTraining XGBoost...")

xgb_model = XGBRegressor(
    n_estimators       = 1000,
    learning_rate      = 0.05,
    max_depth          = 6,
    min_child_weight   = 5,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    reg_alpha          = 0.1,
    reg_lambda         = 1.0,
    random_state       = 42,
    n_jobs             = -1,
    tree_method        = 'hist',
    early_stopping_rounds = 30,
    eval_metric        = 'rmse',
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f"\nBest iteration: {xgb_model.best_iteration}")


# ── 9. Evaluate ──────────────────────────────────────────────

y_pred = xgb_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n===== XGBoost Model Performance =====")
print(f"MAE  : {mae:.4f} km/h")
print(f"RMSE : {rmse:.4f} km/h")
print(f"R²   : {r2:.4f}")


# ── 10. Save Model & Predictions ─────────────────────────────

MODEL_PATH = '/content/drive/MyDrive/ECE1513_Project/Models/xgb_speed_model.pkl'
joblib.dump(xgb_model, MODEL_PATH)
print(f"\nSaved model to {MODEL_PATH}")

preds_df = test_df[['centreline_id', 'direction', 'time_start', TARGET]].copy()
preds_df['predicted_speed'] = y_pred
preds_df.to_csv(
    '/content/drive/MyDrive/ECE1513_Project/Data/xgb_speed_predictions.csv',
    index=False,
)
print("Saved predictions to xgb_speed_predictions.csv")


# ── 11. Feature Importance ───────────────────────────────────

importance_df = pd.DataFrame({
    'feature':    FEATURES,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False)

print("\nTop 20 important features:")
print(importance_df.head(20).to_string())

importance_df.to_csv(
    '/content/drive/MyDrive/ECE1513_Project/Data/xgb_speed_feature_importance.csv',
    index=False,
)
print("\nSaved feature importance to xgb_speed_feature_importance.csv")


# ── 12. Load & Reuse (example) ───────────────────────────────
# import joblib
# xgb_model = joblib.load('/content/drive/MyDrive/ECE1513_Project/Models/xgb_speed_model.pkl')
# y_pred = xgb_model.predict(X_test)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total records loaded: 1279585
Date range: 2020-01-14 00:15:00 to 2024-12-19 23:45:00

Train size: 895709
Val size:   127959
Test size:  255917
Train period: 2020-01-14 00:15:00 → 2023-06-24 22:30:00
Val period:   2023-06-24 22:30:00 → 2023-11-15 12:15:00
Test period:  2023-11-15 12:15:00 → 2024-12-19 23:45:00

Historical avg NaNs in test: speed=204748, volume=204748

After deviation features:
  Train: 895716, Val: 127965, Test: 255904
  train: 895716 → 879707 (dropped 16009 NaN rows)
  val: 127965 → 24443 (dropped 103522 NaN rows)
  test: 255904 → 51167 (dropped 204737 NaN rows)

Training XGBoost...
[0]	validation_0-rmse:11.32782
[50]	validation_0-rmse:5.72984
[96]	validation_0-rmse:5.79365

Best iteration: 66

===== XGBoost Model Performance =====
MAE  : 4.4108 km/h
RMSE : 6.4351 km/h
R²   : 0.6192

Saved model to /content/drive/MyDrive/ECE1513_Project/Model